In [ ]:
# =========================
# CELL 1: setup + install
# =========================

import os
import random
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

# --- Environment detection ---
try:
    from google.colab import drive

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/moses_full_reproduction")
    REPO_DIR = Path("/content/MoSEs")
    REPO_CACHE = DRIVE_ROOT / "repo_cache"
    HF_CACHE = DRIVE_ROOT / "hf_cache"
    RESULTS_ROOT = DRIVE_ROOT / "final_outputs"
else:
    # Local: use the repo directory directly
    REPO_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
    # If running from inside the repo, use the repo root
    if (REPO_DIR / "requirements.txt").exists():
        pass
    elif (REPO_DIR.parent / "requirements.txt").exists():
        REPO_DIR = REPO_DIR.parent
    HF_CACHE = REPO_DIR / ".cache" / "huggingface"
    RESULTS_ROOT = REPO_DIR / "outputs"
    DRIVE_ROOT = None
    REPO_CACHE = None

dirs_to_create = [HF_CACHE, RESULTS_ROOT]
if IN_COLAB:
    dirs_to_create += [DRIVE_ROOT, REPO_CACHE]
for p in dirs_to_create:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def run_cmd(cmd, cwd=None, env=None, shell=False, check=True):
    printable = (
        cmd if isinstance(cmd, str) else " ".join(shlex.quote(str(x)) for x in cmd)
    )
    print("\n" + "=" * 100)
    print(printable)
    print("=" * 100)
    result = subprocess.run(
        cmd, cwd=cwd, env=env, shell=shell, capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode,
            cmd,
            output=result.stdout,
            stderr=result.stderr,
        )
    return result


def copytree_merge(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        s = item
        d = dst / item.name
        if s.is_dir():
            copytree_merge(s, d)
        else:
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(s, d)


if IN_COLAB:
    # Only clone if repo doesn't exist; pull latest otherwise
    if not REPO_DIR.exists():
        run_cmd(
            ["git", "clone", "https://github.com/HuskyDevClub/MoSEs.git", str(REPO_DIR)]
        )
    else:
        print(f"[SKIP] Repo already exists at {REPO_DIR}, pulling latest...")
        run_cmd(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=False)

    # Restore cached outputs only if not already present locally
    for rel in ["weights", "logs"]:
        local_dir = REPO_DIR / rel
        if not local_dir.exists() or not list(local_dir.glob("*")):
            copytree_merge(REPO_CACHE / rel, local_dir)
        else:
            print(f"[SKIP] {rel}/ already exists locally, skipping Drive restore.")

    cached_data_dir = REPO_CACHE / "data"
    if cached_data_dir.exists():
        for p in cached_data_dir.glob("split_datasets*"):
            local_split = REPO_DIR / "data" / p.name
            if not local_split.exists() or not list(local_split.glob("*.json")):
                copytree_merge(p, local_split)
            else:
                print(
                    f"[SKIP] {p.name} already exists locally, skipping Drive restore."
                )

# install dependencies
run_cmd(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
)

# run_cmd(["nvidia-smi"], check=False)

print("Setup complete.")

In [ ]:
# =========================
# CELL 2: configuration + helpers
# =========================

import json
import re
import pandas as pd
from pathlib import Path

DATA_INPUT_DIR = REPO_DIR / "data" / "doc4split"

MAIN_KEYS = ["cmv", "sci", "wp", "xsum"]
LOW_KEYS = ["cnn", "dialogsum", "imdb", "pubmed"]

# You can flip any of these to False if you need to rerun only part of the pipeline.
RUN_FAST = True
RUN_LASTDE = True
RUN_ROBERTA = True

# Official example scripts clearly show main fast/lastde use *_train as reference set.
# The released roberta example points to the unsplit folder; keep this toggle if you want to match that behavior.
ROBERTA_MAIN_USE_FULL_FOLDER = False

DETECTORS = {
    "fast": {
        "enabled": RUN_FAST,
        "crit_type": "fast",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_fast",
        "extra_args": [],
    },
    "lastde": {
        "enabled": RUN_LASTDE,
        "crit_type": "lastde",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_lastde",
        "extra_args": ["--embed_size", "4", "--epsilon", "8", "--tau_prime", "15"],
    },
    "roberta_base": {
        "enabled": RUN_ROBERTA,
        "crit_type": "roberta",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_roberta_base",
        "extra_args": ["--roberta_model_name", "roberta-base-openai-detector"],
    },
}


def sync_outputs_to_drive():
    if not IN_COLAB:
        return
    (REPO_CACHE / "data").mkdir(parents=True, exist_ok=True)
    for p in (REPO_DIR / "data").glob("split_datasets*"):
        copytree_merge(p, REPO_CACHE / "data" / p.name)
    copytree_merge(REPO_DIR / "weights", REPO_CACHE / "weights")
    copytree_merge(REPO_DIR / "logs", REPO_CACHE / "logs")


def dataset_key_from_json_name(name):
    key = Path(name).stem.lower()
    if key.endswith("_dataset"):
        key = key[:-8]
    return key


def subset_json_folder(src_dir, dst_dir, allowed_keys):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in src_dir.glob("*.json"):
        if dataset_key_from_json_name(p.name) in allowed_keys:
            shutil.copy2(p, dst_dir / p.name)


def split_json_folder_dataset_aware(
    src_dir, train_dir, test_dir, main_test_per_label=100, seed=42
):
    """Split each JSON file in src_dir into train/test sets.

    Paper specifies 1800 ref / 200 test for main datasets (100 per label)
    and 200 ref / 200 test for low-resource datasets (half per label).
    """
    src_dir = Path(src_dir)
    train_dir = Path(train_dir)
    test_dir = Path(test_dir)
    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)

    for p in sorted(src_dir.glob("*.json")):
        key = dataset_key_from_json_name(p.name)
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)

        label0 = [x for x in data if int(x["label"]) == 0]
        label1 = [x for x in data if int(x["label"]) == 1]
        rng.shuffle(label0)
        rng.shuffle(label1)

        if key in LOW_KEYS:
            # Paper: 200 ref / 200 test  ->  half per label to test, half to reference
            test_n = min(len(label0), len(label1)) // 2
        else:
            # Paper: 1800 ref / 200 test  ->  100 per label for test
            test_n = min(main_test_per_label, len(label0) // 2, len(label1) // 2)

        test_data = label0[:test_n] + label1[:test_n]
        train_data = label0[test_n:] + label1[test_n:]

        with open(test_dir / p.name, "w", encoding="utf-8") as f:
            json.dump(test_data, f, ensure_ascii=False, indent=2)
        with open(train_dir / p.name, "w", encoding="utf-8") as f:
            json.dump(train_data, f, ensure_ascii=False, indent=2)

        print(
            f"{p.name}: train={len(train_data)}, test={len(test_data)}, test_per_label={test_n}"
        )


def prepare_detector_views(processed_dir):
    """Split the full 8-dataset processed folder into train/test, then create
    main-only and low-only test subsets for evaluation iteration.

    Matches the paper: SAR and CTE both operate on the full 8-class SRR,
    while evaluation iterates over 4 main or 4 low-resource test files.
    """
    processed_dir = Path(processed_dir)

    # Full train/test split (all 8 datasets)
    full_train = Path(str(processed_dir) + "_train")
    full_test = Path(str(processed_dir) + "_test")

    # Test subsets for evaluation iteration only
    main_test = Path(str(processed_dir) + "_main_test")
    low_test = Path(str(processed_dir) + "_low_test")

    # Split full folder into train/test (100 per label for main, half for low)
    if (
        not full_train.exists()
        or not list(full_train.glob("*.json"))
        or not full_test.exists()
        or not list(full_test.glob("*.json"))
    ):
        split_json_folder_dataset_aware(
            processed_dir, full_train, full_test, main_test_per_label=100, seed=42
        )

    # Subset the test folder into main / low for CTE evaluation iteration
    if not main_test.exists() or not list(main_test.glob("*.json")):
        subset_json_folder(full_test, main_test, MAIN_KEYS)
    if not low_test.exists() or not list(low_test.glob("*.json")):
        subset_json_folder(full_test, low_test, LOW_KEYS)

    return {
        "full": processed_dir,  # unsplit, all 8 datasets
        "train": full_train,  # reference set, all 8 datasets
        "test": full_test,  # test set, all 8 datasets
        "main_test": main_test,  # 4 main dataset test files
        "low_test": low_test,  # 4 low-resource dataset test files
    }


def find_best_sar_weights(output_dir):
    """Find the SAR weight file with the highest epoch number (= best val accuracy).

    SAR.py saves a new file each time val_acc improves, so the highest
    epoch number corresponds to the best model.
    """
    output_dir = Path(output_dir)
    pts = list(output_dir.glob("subcentroids_head_epoch*.pt"))
    if not pts:
        return None

    def epoch_num(p):
        m = re.search(r"epoch(\d+)", p.name)
        return int(m.group(1)) if m else 0

    return max(pts, key=epoch_num)


def flatten_json(x, prefix=""):
    out = {}
    if isinstance(x, dict):
        for k, v in x.items():
            out.update(flatten_json(v, f"{prefix}{k}." if prefix else f"{k}."))
    elif isinstance(x, list):
        if len(x) <= 20:
            out[prefix[:-1]] = str(x)
    else:
        out[prefix[:-1]] = x
    return out


print("Helpers ready.")

In [ ]:
# =========================
# CELL 2B: metrics computation helpers
# =========================

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


def compute_cte_metrics(results_base_dir, dataset_keys):
    """Compute accuracy, F1, and AUROC from CTE output JSONs.

    Args:
        results_base_dir: Directory containing detector subdirs with *_results.json files.
        dataset_keys: List of dataset keys to include (e.g., MAIN_KEYS or LOW_KEYS).

    Returns:
        DataFrame with columns: detector, dataset, method, accuracy, f1, auroc
    """
    results_base_dir = Path(results_base_dir)
    records = []

    for det_dir in sorted(results_base_dir.iterdir()):
        if not det_dir.is_dir():
            continue
        det_name = det_dir.name

        for json_file in sorted(det_dir.glob("*_results.json")):
            ds_key = dataset_key_from_json_name(json_file.name.replace("_results", ""))
            if ds_key not in dataset_keys:
                continue

            with open(json_file, "r") as f:
                data = json.load(f)

            for method_name, method_data in data.items():
                y_true = method_data["y_true"]
                y_pred = method_data["y_pred"]
                ai_prob = method_data.get("ai_prob", [])

                acc = accuracy_score(y_true, y_pred)
                f1_val = f1_score(y_true, y_pred, average="binary")

                auroc = None
                try:
                    if ai_prob and len(set(ai_prob)) > 1:
                        auroc = roc_auc_score(y_true, ai_prob)
                except Exception:
                    pass

                records.append(
                    {
                        "detector": det_name,
                        "dataset": ds_key,
                        "method": method_name,
                        "accuracy": acc,
                        "f1": f1_val,
                        "auroc": auroc,
                    }
                )

    return pd.DataFrame(records)


# Paper method names for display — match Table 1/2 naming
METHOD_LABELS = {
    "full_constant": "Static Threshold",
    "full_count": "Nearest Voting",
    "constant": "SAR + Static Threshold",
    "count": "SAR + Nearest Voting",
    "logistic": "MoSEs-lr",
    "xg": "MoSEs-xg",
}
METHOD_ORDER = list(METHOD_LABELS.keys())


def display_results_table(df, metric, title):
    """Pivot metrics DataFrame into a paper-style comparison table."""
    rows = []
    for det in sorted(df["detector"].unique()):
        for method in METHOD_ORDER:
            subset = df[(df["detector"] == det) & (df["method"] == method)]
            if subset.empty:
                continue
            row = {"Detector": det, "Method": METHOD_LABELS[method]}
            for _, r in subset.iterrows():
                row[r["dataset"].upper()] = r[metric]
            numeric_vals = [
                v for v in row.values() if isinstance(v, (int, float)) and pd.notna(v)
            ]
            row["Avg"] = sum(numeric_vals) / len(numeric_vals) if numeric_vals else None
            rows.append(row)

    table = pd.DataFrame(rows)
    print(f"\n{'='*80}")
    print(f"  {title} ({metric})")
    print(f"{'='*80}")
    print(table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return table


def compute_h1_improvement(df):
    """H1: MoSEs-xg vs Static Threshold, averaged across all detectors.

    Paper claims 11.34% average accuracy improvement of MoSEs-xg over
    static threshold across all 3 score models.
    """
    rows = []
    for det in sorted(df["detector"].unique()):
        det_df = df[df["detector"] == det]
        xg_acc = det_df[det_df["method"] == "xg"]["accuracy"].mean()
        st_acc = det_df[det_df["method"] == "full_constant"]["accuracy"].mean()
        pct = ((xg_acc - st_acc) / st_acc * 100) if st_acc > 0 else 0.0
        rows.append(
            {
                "detector": det,
                "MoSEs-xg": xg_acc,
                "Static Threshold": st_acc,
                "Improvement %": pct,
            }
        )
    summary = pd.DataFrame(rows)
    avg_pct = summary["Improvement %"].mean()
    return summary, avg_pct


def compute_h2_improvement(df):
    """H2: MoSEs-xg vs Static Threshold in low-resource, per detector.

    Paper claims 19.43% average across 3 models, 39.15% for RoBERTa.
    """
    return compute_h1_improvement(df)  # same formula, different data


print("Metrics helpers ready.")

In [ ]:
# =========================
# CELL 3: verify input data + build processed folders for all detectors
# =========================

csvs = sorted(DATA_INPUT_DIR.glob("*.csv"))
print("Found CSV files:")
for p in csvs:
    print(" -", p.name)

assert len(csvs) > 0, f"No CSV files found in {DATA_INPUT_DIR}"

# Count the actual number of unique datasets in the input CSVs
# so the skip threshold matches reality (e.g. 4 for main-only, 8 for full)
import csv as _csv

_expected_datasets = set()
for _csv_path in csvs:
    with open(_csv_path, "r", encoding="utf-8") as _f:
        reader = _csv.DictReader(_f)
        for row in reader:
            src = row.get("src", "")
            idx = src.find("_")
            name = src[:idx].lower() if idx != -1 else src.lower()
            if name:
                _expected_datasets.add(name)
EXPECTED_NUM_DATASETS = len(_expected_datasets)
print(f"Expected datasets ({EXPECTED_NUM_DATASETS}): {sorted(_expected_datasets)}")


def build_processed_folder(det_name, cfg):
    out_dir = REPO_DIR / "data" / cfg["out_name"]
    jsons = list(out_dir.glob("*.json")) if out_dir.exists() else []
    if len(jsons) >= EXPECTED_NUM_DATASETS:
        print(f"[SKIP] {det_name}: found {len(jsons)} processed JSONs in {out_dir}")
        return out_dir

    cmd = [
        sys.executable,
        "split_datasets.py",
        "--input_directory",
        str(DATA_INPUT_DIR),
        "--output_directory",
        str(out_dir),
        "--embedding_type",
        "encode",
        "--embedding_model_name",
        "BAAI/bge-m3",
        "--scoring_model_name",
        "EleutherAI/gpt-neo-2.7B",
        "--batch_size",
        "500",
        "--crit_type",
        cfg["crit_type"],
        "--cache_dir",
        str(HF_CACHE),
    ] + cfg["extra_args"]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()
    return out_dir


PROCESSED_DIRS = {}

for det_name, cfg in DETECTORS.items():
    if not cfg["enabled"]:
        continue
    PROCESSED_DIRS[det_name] = build_processed_folder(det_name, cfg)

print("\nProcessed dirs:")
for k, v in PROCESSED_DIRS.items():
    print(k, "->", v)

In [ ]:
# =========================
# CELL 4: create train/test splits and test subsets for all detectors
# =========================

VIEW_PATHS = {}

for det_name, processed_dir in PROCESSED_DIRS.items():
    print(f"\nPreparing views for {det_name}")
    VIEW_PATHS[det_name] = prepare_detector_views(processed_dir)

sync_outputs_to_drive()

for det_name, views in VIEW_PATHS.items():
    print(f"\n=== {det_name} ===")
    for k, v in views.items():
        count = len(list(Path(v).glob("*.json"))) if Path(v).exists() else 0
        print(f"{k:>12}: {v} ({count} json files)")

In [ ]:
# =========================
# CELL 5: train SAR on the full 8-class SRR (matching the paper)
# =========================

# Paper Section 3.3: SAR operates on the full SRR containing all 8 stylistic categories.
# The original run_train.sh trains SAR on the complete (unsplit) fast-processed folder.
# We train on the _train split (all 8 datasets) so test data is excluded.

SAR_OUT = REPO_DIR / "weights" / "model_output_c16"


def train_sar_if_needed(dataset_folder, output_dir):
    class_path = output_dir / "class_names.json"
    weight_path = find_best_sar_weights(output_dir)

    if weight_path is not None and class_path.exists():
        print(f"[SKIP] SAR already exists at {output_dir} (best: {weight_path.name})")
        return weight_path, class_path

    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "SAR.py",
        "train",
        "--model_name",
        "BAAI/bge-m3",
        "--datasets_folder",
        str(dataset_folder),
        "--embedding_type",
        "encode",
        "--num_epochs",
        "100",
        "--batch_size",
        "32",
        "--num_subcentroids",
        "4",
        "--output_dir",
        str(output_dir),
    ]

    # If a checkpoint exists from a previous interrupted run, resume from it
    checkpoint_path = output_dir / "checkpoint_latest.pt"
    if checkpoint_path.exists():
        print(f"[RESUME] Found checkpoint at {checkpoint_path}, resuming training...")
        cmd += ["--resume_checkpoint", str(checkpoint_path)]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()

    weight_path = find_best_sar_weights(output_dir)
    assert (
        weight_path is not None
    ), f"No SAR weights found in {output_dir} after training"
    return weight_path, class_path


assert (
    "fast" in VIEW_PATHS
), "fast detector processed folder is needed to train SAR as in the released examples."

# Train a single SAR on the full _train folder (all 8 datasets)
SAR_PATH, SAR_CLASS = train_sar_if_needed(VIEW_PATHS["fast"]["train"], SAR_OUT)

print("SAR weights:", SAR_PATH)
print("SAR classes:", SAR_CLASS)

In [ ]:
# =========================
# CELL 6: run CTE evaluation for all detectors on main + low-resource
# =========================

# Paper: CTE uses the full SRR (all 8 datasets) as the reference pool.
# Original run_cte_fast_detect.sh uses the _train folder (all 8 datasets).
# Original run_cte_roberta_base.sh uses the unsplit folder (all 8 datasets).

DONE_DIR = REPO_DIR / "logs" / "done_markers"
DONE_DIR.mkdir(parents=True, exist_ok=True)


def run_cte_single(
    test_file, result_prefix, datasets_folder, sar_path, class_path, marker_prefix
):
    """Run CTE.py on a single test file. Returns (marker_name, success, error)."""
    test_file = Path(test_file)
    result_prefix = Path(result_prefix)
    result_prefix.parent.mkdir(parents=True, exist_ok=True)
    marker = DONE_DIR / f"{marker_prefix}__{test_file.stem}.done"

    if marker.exists():
        return marker.name, True, "skipped"

    cmd = [
        sys.executable,
        "CTE.py",
        "--file_path",
        str(test_file),
        "--result_file_path",
        str(result_prefix),
        "--datasets_folder",
        str(datasets_folder),
        "--sar_path",
        str(sar_path),
        "--class_path",
        str(class_path),
    ]
    try:
        result = subprocess.run(
            cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=None, text=True
        )
        if result.returncode != 0:
            return marker.name, False, f"exit code {result.returncode}"
        marker.touch()
        return marker.name, True, result.stdout[-200:] if result.stdout else ""
    except Exception as e:
        return marker.name, False, str(e)


# Collect all CTE jobs
cte_jobs = []

for det_name, views in VIEW_PATHS.items():
    if det_name == "roberta_base" and ROBERTA_MAIN_USE_FULL_FOLDER:
        ref_folder = views["full"]
    else:
        ref_folder = views["train"]

    # Main test files
    main_test_dir = Path(views["main_test"])
    for test_file in sorted(main_test_dir.glob("*.json")):
        cte_jobs.append(
            {
                "test_file": test_file,
                "result_prefix": REPO_DIR / "logs" / "reproduction" / "main" / det_name,
                "datasets_folder": ref_folder,
                "sar_path": SAR_PATH,
                "class_path": SAR_CLASS,
                "marker_prefix": f"main__{det_name}",
                "label": f"{det_name}/main/{test_file.stem}",
            }
        )

    # Low-resource test files
    low_test_dir = Path(views["low_test"])
    for test_file in sorted(low_test_dir.glob("*.json")):
        cte_jobs.append(
            {
                "test_file": test_file,
                "result_prefix": REPO_DIR / "logs" / "reproduction" / "low" / det_name,
                "datasets_folder": views["train"],
                "sar_path": SAR_PATH,
                "class_path": SAR_CLASS,
                "marker_prefix": f"low__{det_name}",
                "label": f"{det_name}/low/{test_file.stem}",
            }
        )

print(f"Total CTE jobs: {len(cte_jobs)}")

# Run jobs sequentially (CTE is CPU-bound; parallelism just causes contention)
completed = 0
failed = []

for job in cte_jobs:
    marker_name, success, msg = run_cte_single(
        job["test_file"],
        job["result_prefix"],
        job["datasets_folder"],
        job["sar_path"],
        job["class_path"],
        job["marker_prefix"],
    )
    completed += 1
    status = "OK" if success else "FAIL"
    print(f"[{completed}/{len(cte_jobs)}] {status}: {job['label']} ({msg[:80]})")
    if not success:
        failed.append((job["label"], msg))

sync_outputs_to_drive()

if failed:
    print(f"\n{len(failed)} jobs FAILED:")
    for label, msg in failed:
        print(f"  {label}: {msg[:200]}")
else:
    print(f"\nAll {len(cte_jobs)} CTE jobs finished successfully.")

In [ ]:
# =========================
# CELL 7: compute and display main results (Table 1)
# =========================

main_results_dir = REPO_DIR / "logs" / "reproduction" / "main"
main_metrics_df = compute_cte_metrics(main_results_dir, MAIN_KEYS)

if main_metrics_df.empty:
    print("No main results found. Run Cell 6 (CTE evaluation) first.")
else:
    # Table 1: Detection performance
    acc_table = display_results_table(
        main_metrics_df, "accuracy", "Table 1: Main Results"
    )
    f1_table = display_results_table(main_metrics_df, "f1", "Table 1: Main Results")

    # AUROC (only for methods with probability outputs)
    auroc_df = main_metrics_df[main_metrics_df["auroc"].notna()]
    if not auroc_df.empty:
        auroc_table = display_results_table(auroc_df, "auroc", "Table 1: Main Results")

    # H1: MoSEs-xg vs Static Threshold (paper claims +11.34% avg across 3 models)
    h1_summary, h1_avg_pct = compute_h1_improvement(main_metrics_df)
    print(f"\n{'='*80}")
    print(f"  H1 Verification: MoSEs-xg vs Static Threshold (Accuracy)")
    print(f"{'='*80}")
    print(h1_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"\n  Average improvement across detectors: {h1_avg_pct:+.2f}%")
    print(f"  Paper claims:                         +11.34%")

    # Save detailed metrics CSV
    main_metrics_csv = RESULTS_ROOT / "main_metrics_detailed.csv"
    main_metrics_df.to_csv(main_metrics_csv, index=False)
    print(f"\nDetailed metrics saved to: {main_metrics_csv}")

# Fix timestamps before 1980 (ZIP format requirement)
import time

_min_ts = time.mktime((1980, 1, 1, 0, 0, 0, 0, 0, -1))
for _archive_dir in [REPO_DIR / "logs", REPO_DIR / "weights"]:
    if _archive_dir.exists():
        for _p in _archive_dir.rglob("*"):
            if _p.stat().st_mtime < _min_ts:
                os.utime(_p, (_min_ts, _min_ts))

# Archive logs and weights
logs_zip = shutil.make_archive(
    str(RESULTS_ROOT / "moses_logs"), "zip", REPO_DIR / "logs"
)
weights_zip = shutil.make_archive(
    str(RESULTS_ROOT / "moses_weights"), "zip", REPO_DIR / "weights"
)
print(f"\nLogs zip   : {logs_zip}")
print(f"Weights zip: {weights_zip}")

sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 8: download and stage low-resource data from MAGE benchmark
# =========================
#
# The paper (Wu et al., 2025) states: "We constructed 8 datasets with different
# styles mainly based on MAGE (Li et al., 2024)." The 4 low-resource datasets
# (CNN, DialogSum, IMDB, PubMedQA) each contain 200 human-written + 200 GPT-4
# generated samples. The original MoSEs repo only ships the main dataset CSV,
# so we download the low-resource data from the MAGE benchmark (yaful/MAGE)
# on HuggingFace.
#
# MAGE src values: cnn_human, cnn_gpt4, dialogsum_human, dialogsum_gpt4,
#                  imdb_human, imdb_gpt4, pubmed_human, pubmed_gpt4

import pandas as pd
from pathlib import Path

LOW_INPUT_DIR = REPO_DIR / "data" / "doc4split_low"
LOW_INPUT_DIR.mkdir(parents=True, exist_ok=True)
LOW_CSV = LOW_INPUT_DIR / "low_resource_mage_gpt4.csv"

# Check if data is already staged
if LOW_CSV.exists():
    existing_df = pd.read_csv(LOW_CSV)
    print(
        f"[SKIP] Low-resource CSV already exists: {LOW_CSV} ({len(existing_df)} rows)"
    )
else:
    # Try local copy first (doc4split_tiny already has this data)
    TINY_CSV = REPO_DIR / "data" / "doc4split_tiny" / "tiny_gpt4_200.csv"

    if TINY_CSV.exists():
        print(f"Found existing low-resource data at {TINY_CSV}")
        df = pd.read_csv(TINY_CSV)
    else:
        print("Downloading low-resource data from MAGE benchmark (yaful/MAGE)...")
        from datasets import load_dataset

        ds = load_dataset("yaful/MAGE", split="test")
        mage_df = ds.to_pandas()

        # Filter to the 4 low-resource domains with GPT-4 generated + human text
        # MAGE uses these src values for the low-resource domains:
        #   cnn_human, cnn_gpt4, dialogsum_human, dialogsum_gpt4,
        #   imdb_human, imdb_gpt4, pubmed_human, pubmed_gpt4
        low_src_values = [
            "cnn_human",
            "cnn_gpt4",
            "dialogsum_human",
            "dialogsum_gpt4",
            "imdb_human",
            "imdb_gpt4",
            "pubmed_human",
            "pubmed_gpt4",
        ]
        df = mage_df[mage_df["src"].isin(low_src_values)].copy()

        # MAGE label convention: 0 = machine, 1 = human
        # MoSEs CSV convention: 0 = human, 1 = machine (matches main CSV)
        # Check which convention the main CSV uses
        main_csv = REPO_DIR / "data" / "doc4split" / "filtered_train_main_1000.csv"
        main_df = pd.read_csv(main_csv, nrows=5, encoding="utf-8-sig")
        main_human_label = main_df[main_df["src"].str.contains("human")]["label"].iloc[
            0
        ]
        mage_human_label = df[df["src"].str.contains("human")]["label"].iloc[0]
        if main_human_label != mage_human_label:
            print(
                f"Flipping labels: MAGE human={mage_human_label}, main CSV human={main_human_label}"
            )
            df["label"] = 1 - df["label"]

        # Sample exactly 200 per src to match paper specification
        sampled = []
        for src_val in low_src_values:
            subset = df[df["src"] == src_val]
            if len(subset) >= 200:
                sampled.append(subset.sample(n=200, random_state=42))
            else:
                print(
                    f"  WARNING: {src_val} has only {len(subset)} samples (need 200), using all"
                )
                sampled.append(subset)
        df = pd.concat(sampled, ignore_index=True)

    # Ensure correct columns
    required_cols = {"text", "label", "src"}
    assert required_cols.issubset(
        df.columns
    ), f"Missing columns: {required_cols - set(df.columns)}"

    # Save to staging directory
    df[["text", "label", "src"]].to_csv(LOW_CSV, index=False)
    print(f"Saved {len(df)} rows to {LOW_CSV}")

# Verify
df = pd.read_csv(LOW_CSV)
print(f"\nStaged low-resource CSV: {LOW_CSV}")
print(f"Total rows: {len(df)}")
print(f"\nPer-source counts:")
for src_val in sorted(df["src"].unique()):
    count = len(df[df["src"] == src_val])
    label_dist = df[df["src"] == src_val]["label"].value_counts().to_dict()
    print(f"  {src_val}: {count} rows (labels: {label_dist})")

prefixes = df["src"].str.split("_").str[0].unique().tolist()
print(f"\nDetected prefixes: {sorted(prefixes)}")
print(f"Expected LOW_KEYS: {sorted(LOW_KEYS)}")
missing = set(LOW_KEYS) - set(prefixes)
assert not missing, f"Missing prefixes: {sorted(missing)}"
print("\nLow-resource data is staged and ready.")

In [ ]:
# =========================
# CELL 8A: stage + validate low-resource raw CSVs
# =========================

import pandas as pd
import shutil
from pathlib import Path

LOW_INPUT_DIR = REPO_DIR / "data" / "doc4split_low"
LOW_INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Only copy from Drive if doc4split_low has no CSVs yet (Cell 8 may have already staged them)
existing = sorted(LOW_INPUT_DIR.glob("*.csv"))

if existing:
    print(f"[SKIP] {len(existing)} CSV(s) already in {LOW_INPUT_DIR}, not overwriting.")
elif IN_COLAB:
    LOW_RAW_SRC_DIR = DRIVE_ROOT / "low_resource_csvs"
    src_csvs = sorted(LOW_RAW_SRC_DIR.glob("*.csv")) if LOW_RAW_SRC_DIR.exists() else []
    assert src_csvs, (
        f"No low-resource CSV files found in {LOW_RAW_SRC_DIR} and none in {LOW_INPUT_DIR}.\n"
        "Either run Cell 8 first to download from MAGE, or place your CSVs in the Drive folder."
    )
    for p in src_csvs:
        shutil.copy2(p, LOW_INPUT_DIR / p.name)
else:
    # Local: expect CSVs already placed in doc4split_low
    print(
        f"WARNING: No CSV files found in {LOW_INPUT_DIR}.\n"
        f"Place your low-resource raw CSV(s) there before running this cell."
    )

print("Staged low-resource CSV files:")
for p in sorted(LOW_INPUT_DIR.glob("*.csv")):
    print(" -", p.name)

# Validate that the staged CSVs actually contain the 4 low-resource dataset prefixes
required_low = set(LOW_KEYS)
seen_low = set()

for csv_path in sorted(LOW_INPUT_DIR.glob("*.csv")):
    df = pd.read_csv(csv_path)
    missing_cols = {"src", "text", "label"} - set(df.columns)
    assert not missing_cols, f"{csv_path.name} is missing columns: {missing_cols}"

    prefixes = (
        df["src"]
        .dropna()
        .astype(str)
        .map(lambda s: s.split("_", 1)[0].lower())
        .unique()
        .tolist()
    )
    seen_low.update(prefixes)

print("\nDetected low-resource src prefixes:", sorted(seen_low))
missing_low = required_low - seen_low
assert not missing_low, (
    f"Missing low-resource dataset prefixes in staged CSVs: {sorted(missing_low)}\n"
    f"Need all of: {sorted(required_low)}"
)

print("\nLow-resource raw data is staged and validated.")

In [ ]:
# =========================
# CELL 8B: build processed low-resource folders for all detectors
# =========================

LOW_DETECTORS = {
    "fast": {
        "enabled": RUN_FAST,
        "crit_type": "fast",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_low_200_fast",
        "extra_args": [],
    },
    "lastde": {
        "enabled": RUN_LASTDE,
        "crit_type": "lastde",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_low_200_lastde",
        "extra_args": ["--embed_size", "4", "--epsilon", "8", "--tau_prime", "15"],
    },
    "roberta_base": {
        "enabled": RUN_ROBERTA,
        "crit_type": "roberta",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_low_200_roberta_base",
        "extra_args": ["--roberta_model_name", "roberta-base-openai-detector"],
    },
}


def has_expected_jsons(folder, expected_keys):
    folder = Path(folder)
    if not folder.exists():
        return False
    found = {dataset_key_from_json_name(p.name) for p in folder.glob("*.json")}
    return set(expected_keys).issubset(found)


def build_low_processed_folder(det_name, cfg):
    out_dir = REPO_DIR / "data" / cfg["out_name"]

    if has_expected_jsons(out_dir, LOW_KEYS):
        print(f"[SKIP] {det_name}: found expected low-resource JSONs in {out_dir}")
        return out_dir

    if out_dir.exists():
        shutil.rmtree(out_dir)

    cmd = [
        sys.executable,
        "split_datasets.py",
        "--input_directory",
        str(LOW_INPUT_DIR),
        "--output_directory",
        str(out_dir),
        "--embedding_type",
        "encode",
        "--embedding_model_name",
        "BAAI/bge-m3",
        "--scoring_model_name",
        "EleutherAI/gpt-neo-2.7B",
        "--batch_size",
        "500",
        "--crit_type",
        cfg["crit_type"],
        "--cache_dir",
        str(HF_CACHE),
    ] + cfg["extra_args"]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()

    found = sorted(p.name for p in out_dir.glob("*.json"))
    print(f"\nProcessed low-resource JSONs for {det_name}:")
    for name in found:
        print(" -", name)

    assert has_expected_jsons(
        out_dir, LOW_KEYS
    ), f"{det_name}: expected low-resource JSONs were not all created in {out_dir}"
    return out_dir


LOW_PROCESSED_DIRS = {}

for det_name, cfg in LOW_DETECTORS.items():
    if not cfg["enabled"]:
        continue
    LOW_PROCESSED_DIRS[det_name] = build_low_processed_folder(det_name, cfg)

print("\nLOW_PROCESSED_DIRS:")
for k, v in LOW_PROCESSED_DIRS.items():
    print(k, "->", v)

In [ ]:
# =========================
# CELL 8C: create low-resource train/test splits
# =========================


def prepare_low_detector_views(processed_dir):
    """
    Low-resource-only pipeline:
    - processed_dir contains only cnn/dialogsum/imdb/pubmed JSONs
    - split_json_folder_dataset_aware will create 200 ref / 200 test per dataset
      when each JSON has 400 rows balanced by label
    """
    processed_dir = Path(processed_dir)

    full_train = Path(str(processed_dir) + "_train")
    full_test = Path(str(processed_dir) + "_test")
    low_test = Path(str(processed_dir) + "_low_test")

    need_split = (
        not full_train.exists()
        or not full_test.exists()
        or not has_expected_jsons(full_train, LOW_KEYS)
        or not has_expected_jsons(full_test, LOW_KEYS)
    )
    if need_split:
        if full_train.exists():
            shutil.rmtree(full_train)
        if full_test.exists():
            shutil.rmtree(full_test)
        split_json_folder_dataset_aware(
            processed_dir,
            full_train,
            full_test,
            main_test_per_label=100,  # ignored for LOW_KEYS; function uses half-per-label
            seed=42,
        )

    if low_test.exists():
        shutil.rmtree(low_test)
    subset_json_folder(full_test, low_test, LOW_KEYS)

    return {
        "full": processed_dir,
        "train": full_train,
        "test": full_test,
        "low_test": low_test,
    }


LOW_VIEW_PATHS = {}

for det_name, processed_dir in LOW_PROCESSED_DIRS.items():
    print(f"\nPreparing low-resource views for {det_name}")
    LOW_VIEW_PATHS[det_name] = prepare_low_detector_views(processed_dir)

sync_outputs_to_drive()

for det_name, views in LOW_VIEW_PATHS.items():
    print(f"\n=== LOW {det_name} ===")
    for k, v in views.items():
        count = len(list(Path(v).glob("*.json"))) if Path(v).exists() else 0
        print(f"{k:>10}: {v} ({count} json files)")

In [ ]:
# =========================
# CELL 8D: train a separate SAR for the low-resource SRR
# =========================

LOW_SAR_OUT = REPO_DIR / "weights" / "model_output_low_c4"

assert (
    "fast" in LOW_VIEW_PATHS
), "fast low-resource processed folder is needed to train low-resource SAR."

LOW_SAR_PATH, LOW_SAR_CLASS = train_sar_if_needed(
    LOW_VIEW_PATHS["fast"]["train"],
    LOW_SAR_OUT,
)

print("LOW_SAR_PATH :", LOW_SAR_PATH)
print("LOW_SAR_CLASS:", LOW_SAR_CLASS)
sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 8E: run CTE evaluation on low-resource datasets only
# =========================

DONE_DIR = REPO_DIR / "logs" / "done_markers"
DONE_DIR.mkdir(parents=True, exist_ok=True)


def run_cte_folder_safe(
    test_folder, datasets_folder, result_prefix, sar_path, class_path, marker_prefix
):
    test_folder = Path(test_folder)
    result_prefix = Path(result_prefix)
    result_prefix.mkdir(parents=True, exist_ok=True)

    test_files = sorted(test_folder.glob("*.json"))
    assert test_files, f"No test JSON files found in {test_folder}"

    for test_file in test_files:
        marker = DONE_DIR / f"{marker_prefix}__{test_file.stem}.done"
        if marker.exists():
            print(f"[SKIP] {marker.name}")
            continue

        cmd = [
            sys.executable,
            "CTE.py",
            "--file_path",
            str(test_file),
            "--result_file_path",
            str(result_prefix),
            "--datasets_folder",
            str(datasets_folder),
            "--sar_path",
            str(sar_path),
            "--class_path",
            str(class_path),
        ]
        result = subprocess.run(
            cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=None, text=True
        )
        if result.returncode != 0:
            raise subprocess.CalledProcessError(
                result.returncode, cmd, output=result.stdout
            )
        marker.touch()
        sync_outputs_to_drive()


for det_name, views in LOW_VIEW_PATHS.items():
    print(f"\n==================== {det_name}: LOW-RESOURCE ====================")

    # use the clean 200-reference split for all detectors in the low-resource run
    ref_folder = views["train"]

    run_cte_folder_safe(
        test_folder=views["low_test"],
        datasets_folder=ref_folder,
        result_prefix=REPO_DIR / "logs" / "reproduction" / "low_only" / det_name,
        sar_path=LOW_SAR_PATH,
        class_path=LOW_SAR_CLASS,
        marker_prefix=f"low_only__{det_name}",
    )

print("All low-resource CTE runs finished.")
sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 8F: compute and display low-resource results (Table 2)
# =========================

low_results_dir = REPO_DIR / "logs" / "reproduction" / "low_only"
low_metrics_df = compute_cte_metrics(low_results_dir, LOW_KEYS)

if low_metrics_df.empty:
    print("No low-resource results found. Run Cell 8E first.")
else:
    # Table 2: Low-resource detection performance
    acc_table_low = display_results_table(
        low_metrics_df, "accuracy", "Table 2: Low-Resource Results"
    )
    f1_table_low = display_results_table(
        low_metrics_df, "f1", "Table 2: Low-Resource Results"
    )

    # AUROC (only for methods with probability outputs)
    auroc_low = low_metrics_df[low_metrics_df["auroc"].notna()]
    if not auroc_low.empty:
        auroc_table_low = display_results_table(
            auroc_low, "auroc", "Table 2: Low-Resource Results"
        )

    # H2: MoSEs-xg vs Static Threshold per detector (low-resource)
    # Paper claims: 19.43% average across 3 models, 39.15% for RoBERTa
    h2_summary, h2_avg_pct = compute_h2_improvement(low_metrics_df)
    print(f"\n{'='*80}")
    print(f"  H2 Verification: MoSEs-xg vs Static Threshold - Low-Resource (Accuracy)")
    print(f"{'='*80}")
    print(h2_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"\n  Average improvement across detectors: {h2_avg_pct:+.2f}%")
    print(f"  Paper claims (avg across 3 models):   +19.43%")
    print(f"  Paper claims (RoBERTa specifically):   +39.15%")

    # Save detailed metrics CSV
    low_metrics_csv = RESULTS_ROOT / "low_resource_metrics_detailed.csv"
    low_metrics_df.to_csv(low_metrics_csv, index=False)
    print(f"\nDetailed metrics saved to: {low_metrics_csv}")

sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 9A: run CTE ablation experiments (Tables 4 & 5)
# =========================

# Table 4: SAR ablation (--no_router)
# Table 5: Per-feature conditional ablation (--remove_cond_idx)
#   Index mapping: 0=text_len, 1=logprob_mean, 2=logprob_var,
#                  3=2gram_rep, 4=3gram_rep, 5=ttr
# Additional coarse ablations: --no_embedding, --no_condition, --pca_dim -1

COND_FEATURE_NAMES = {
    0: "text_length",
    1: "logprob_mean",
    2: "logprob_var",
    3: "2gram_rep",
    4: "3gram_rep",
    5: "ttr",
}

ABLATIONS = {
    # Table 4: SAR ablation
    "no_router": ["--no_router"],
    # Table 5: coarse ablations
    "no_embedding": ["--no_embedding"],
    "no_condition": ["--no_condition"],
    "no_pca": ["--pca_dim", "-1"],
    # Table 5: per-feature ablations (w.o. each conditional feature)
    **{
        f"wo_{name}": ["--remove_cond_idx", str(idx)]
        for idx, name in COND_FEATURE_NAMES.items()
    },
}

ablation_jobs = []

for abl_name, abl_flags in ABLATIONS.items():
    for det_name, views in VIEW_PATHS.items():
        if det_name == "roberta_base" and ROBERTA_MAIN_USE_FULL_FOLDER:
            ref_folder = views["full"]
        else:
            ref_folder = views["train"]

        main_test_dir = Path(views["main_test"])
        for test_file in sorted(main_test_dir.glob("*.json")):
            marker_prefix = f"abl_{abl_name}__{det_name}"
            marker = DONE_DIR / f"{marker_prefix}__{test_file.stem}.done"

            if marker.exists():
                continue

            ablation_jobs.append(
                {
                    "test_file": test_file,
                    "result_dir": REPO_DIR
                    / "logs"
                    / "reproduction"
                    / "ablation"
                    / abl_name
                    / det_name,
                    "datasets_folder": ref_folder,
                    "sar_path": SAR_PATH,
                    "class_path": SAR_CLASS,
                    "marker_prefix": marker_prefix,
                    "abl_flags": abl_flags,
                    "label": f"{abl_name}/{det_name}/{test_file.stem}",
                }
            )

print(f"Ablation CTE jobs to run: {len(ablation_jobs)} (already-done jobs skipped)")

if ablation_jobs:

    def run_ablation_cte(job):
        test_file = Path(job["test_file"])
        result_dir = Path(job["result_dir"])
        result_dir.mkdir(parents=True, exist_ok=True)
        marker = DONE_DIR / f"{job['marker_prefix']}__{test_file.stem}.done"

        cmd = [
            sys.executable,
            "CTE.py",
            "--file_path",
            str(test_file),
            "--result_file_path",
            str(result_dir),
            "--datasets_folder",
            str(job["datasets_folder"]),
            "--sar_path",
            str(job["sar_path"]),
            "--class_path",
            str(job["class_path"]),
        ] + job["abl_flags"]

        try:
            result = subprocess.run(
                cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=None, text=True
            )
            if result.returncode != 0:
                return job["label"], False, f"exit code {result.returncode}"
            marker.touch()
            return job["label"], True, "ok"
        except Exception as e:
            return job["label"], False, str(e)

    completed = 0
    failed = []

    for job in ablation_jobs:
        lbl, success, msg = run_ablation_cte(job)
        completed += 1
        status = "OK" if success else "FAIL"
        print(f"[{completed}/{len(ablation_jobs)}] {status}: {lbl} ({msg[:60]})")
        if not success:
            failed.append((lbl, msg))

    sync_outputs_to_drive()

    if failed:
        print(f"\n{len(failed)} ablation jobs FAILED:")
        for lbl, msg in failed:
            print(f"  {lbl}: {msg[:200]}")
    else:
        print(f"\nAll {len(ablation_jobs)} ablation jobs finished.")
else:
    print("All ablation jobs already done (skipped).")

In [ ]:
# =========================
# CELL 9B: compute and display ablation results (Tables 4 & 5)
# =========================

ablation_base = REPO_DIR / "logs" / "reproduction" / "ablation"

# Collect metrics for each ablation and the full (non-ablated) main results
all_ablation_rows = []

# Full model (no ablation) as reference
main_df = compute_cte_metrics(REPO_DIR / "logs" / "reproduction" / "main", MAIN_KEYS)
if not main_df.empty:
    for _, r in main_df.iterrows():
        all_ablation_rows.append(
            {
                "ablation": "full_model",
                "detector": r["detector"],
                "dataset": r["dataset"],
                "method": r["method"],
                "accuracy": r["accuracy"],
                "f1": r["f1"],
            }
        )

# Each ablation variant
for abl_name in ABLATIONS:
    abl_dir = ablation_base / abl_name
    if not abl_dir.exists():
        continue
    abl_df = compute_cte_metrics(abl_dir, MAIN_KEYS)
    for _, r in abl_df.iterrows():
        all_ablation_rows.append(
            {
                "ablation": abl_name,
                "detector": r["detector"],
                "dataset": r["dataset"],
                "method": r["method"],
                "accuracy": r["accuracy"],
                "f1": r["f1"],
            }
        )

abl_combined = pd.DataFrame(all_ablation_rows)

if abl_combined.empty:
    print("No ablation results found. Run Cell 9A first.")
else:
    # ── Table 4: SAR ablation (H4) ──
    # Paper: "we show the average results with Lastde on the main datasets"
    # Compare all 4 threshold methods with vs without SAR, Lastde only.
    #
    # Mapping:
    #   "w.o. SAR" Static Threshold   = full_model full_constant
    #   "w.o. SAR" Nearest Voting     = full_model full_count
    #   "w.o. SAR" Logistic Regression = no_router logistic
    #   "w.o. SAR" XGBoost            = no_router xg
    #   "w. SAR"   Static Threshold   = full_model constant
    #   "w. SAR"   Nearest Voting     = full_model count
    #   "w. SAR"   MoSEs-lr           = full_model logistic
    #   "w. SAR"   MoSEs-xg           = full_model xg

    lastde_full = abl_combined[
        (abl_combined["ablation"] == "full_model")
        & (abl_combined["detector"] == "lastde")
    ]
    lastde_no_router = abl_combined[
        (abl_combined["ablation"] == "no_router")
        & (abl_combined["detector"] == "lastde")
    ]

    if not lastde_full.empty:
        print(f"\n{'='*80}")
        print("  Table 4: SAR Ablation (H4) — Lastde, averaged across main datasets")
        print(f"{'='*80}")

        t4_rows = []
        # w.o. SAR rows
        for method, label in [
            ("full_constant", "Static Threshold"),
            ("full_count", "Nearest Voting"),
        ]:
            subset = lastde_full[lastde_full["method"] == method]
            if not subset.empty:
                t4_rows.append(
                    {
                        "SAR": "w.o. SAR",
                        "Threshold Estimator": label,
                        "Avg. Accuracy": subset["accuracy"].mean(),
                        "Avg. F1 score": subset["f1"].mean(),
                    }
                )
        for method, label in [
            ("logistic", "Logistic Regression"),
            ("xg", "XGBoost"),
        ]:
            subset = lastde_no_router[lastde_no_router["method"] == method]
            if not subset.empty:
                t4_rows.append(
                    {
                        "SAR": "w.o. SAR",
                        "Threshold Estimator": label,
                        "Avg. Accuracy": subset["accuracy"].mean(),
                        "Avg. F1 score": subset["f1"].mean(),
                    }
                )

        # w. SAR rows
        for method, label in [
            ("constant", "Static Threshold"),
            ("count", "Nearest Voting"),
            ("logistic", "MoSEs-lr"),
            ("xg", "MoSEs-xg"),
        ]:
            subset = lastde_full[lastde_full["method"] == method]
            if not subset.empty:
                t4_rows.append(
                    {
                        "SAR": "w. SAR",
                        "Threshold Estimator": label,
                        "Avg. Accuracy": subset["accuracy"].mean(),
                        "Avg. F1 score": subset["f1"].mean(),
                    }
                )

        t4_table = pd.DataFrame(t4_rows)
        print(t4_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # ── Table 5: Per-feature conditional ablation (H3) ──
    # Paper: "we show the average results with Lastde on the main datasets"
    # Rows: w.o. text_length, w.o. logprob_mean, ..., w.o. semantic, Default
    per_feature_names = [f"wo_{name}" for name in COND_FEATURE_NAMES.values()]
    all_t5_ablations = (
        ["full_model"] + per_feature_names + ["no_embedding", "no_condition"]
    )
    # Filter to Lastde only and MoSEs methods only (matching paper Table 5)
    t5_data = abl_combined[
        (abl_combined["ablation"].isin(all_t5_ablations))
        & (abl_combined["detector"] == "lastde")
        & (abl_combined["method"].isin(["logistic", "xg"]))
    ]

    if not t5_data.empty:
        print(f"\n{'='*80}")
        print(
            "  Table 5: Conditional Feature Ablation (H3) — Lastde, averaged across main datasets"
        )
        print(f"{'='*80}")

        t5_rows = []
        for abl in all_t5_ablations:
            abl_subset = t5_data[t5_data["ablation"] == abl]
            for method in ["logistic", "xg"]:
                m_subset = abl_subset[abl_subset["method"] == method]
                if m_subset.empty:
                    continue
                label = (
                    "Default"
                    if abl == "full_model"
                    else f"w.o. {abl.replace('wo_', '').replace('no_', 'all ')}"
                )
                t5_rows.append(
                    {
                        "Condition": label,
                        "Method": "MoSEs-lr" if method == "logistic" else "MoSEs-xg",
                        "Avg Accuracy": m_subset["accuracy"].mean(),
                        "Avg F1": m_subset["f1"].mean(),
                    }
                )

        t5_table = pd.DataFrame(t5_rows)
        print(t5_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # Save ablation metrics
    abl_csv = RESULTS_ROOT / "ablation_metrics_detailed.csv"
    abl_combined.to_csv(abl_csv, index=False)
    print(f"\nAblation metrics saved to: {abl_csv}")

sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 10: Additional Experiment 1 — Reference Set Size Sensitivity
# =========================
# The paper evaluates with 1800 (main) and 200 (low-resource) reference samples
# per dataset but does not explore the curve in between. We vary the per-dataset
# reference size and measure detection accuracy to find the "minimum viable
# reference size" for MoSEs.
#
# SAR weights are kept fixed (trained on the full reference set); only the CTE
# reference pool size changes. This isolates CTE's sensitivity to data volume.

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

REF_SIZES_PER_DATASET = [100, 200, 400, 800, 1200, 1800]


def subsample_reference_folder(src_dir, dst_dir, total_per_dataset, seed=42):
    """Create a subsampled reference folder with N total samples per dataset (balanced)."""
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)

    for p in sorted(src_dir.glob("*.json")):
        dst_file = dst_dir / p.name
        if dst_file.exists():
            continue

        with open(p, "r") as f:
            data = json.load(f)

        label0 = [x for x in data if int(x["label"]) == 0]
        label1 = [x for x in data if int(x["label"]) == 1]
        rng.shuffle(label0)
        rng.shuffle(label1)

        n_per_label = total_per_dataset // 2
        n0 = min(n_per_label, len(label0))
        n1 = min(n_per_label, len(label1))
        sampled = label0[:n0] + label1[:n1]

        with open(dst_file, "w") as f:
            json.dump(sampled, f, ensure_ascii=False, indent=2)

    return dst_dir


# Collect all jobs
ref_size_jobs = []
for det_name, views in VIEW_PATHS.items():
    full_train = views["train"]
    main_test_dir = Path(views["main_test"])

    for ref_size in REF_SIZES_PER_DATASET:
        for test_file in sorted(main_test_dir.glob("*.json")):
            ref_size_jobs.append(
                {
                    "det_name": det_name,
                    "ref_size": ref_size,
                    "full_train": full_train,
                    "test_file": test_file,
                }
            )

print(f"Reference size sensitivity jobs: {len(ref_size_jobs)}")

# Run CTE for each reference size
completed = 0
for job in ref_size_jobs:
    det_name = job["det_name"]
    ref_size = job["ref_size"]
    full_train = job["full_train"]
    test_file = job["test_file"]

    # Full size → use original _train; otherwise subsample
    if ref_size >= 1800:
        ref_dir = full_train
    else:
        ref_dir = Path(str(full_train) + f"_sub{ref_size}")
        subsample_reference_folder(full_train, ref_dir, ref_size)

    result_dir = (
        REPO_DIR / "logs" / "reproduction" / "ref_size" / f"size_{ref_size}" / det_name
    )
    marker_prefix = f"refsize_{ref_size}__{det_name}"

    marker_name, success, msg = run_cte_single(
        test_file, result_dir, ref_dir, SAR_PATH, SAR_CLASS, marker_prefix
    )
    completed += 1
    status = "OK" if success else "FAIL"
    print(
        f"[{completed}/{len(ref_size_jobs)}] {status}: "
        f"{det_name}/size_{ref_size}/{test_file.stem} ({msg[:60]})"
    )

sync_outputs_to_drive()

# Collect results
ref_size_records = []
for ref_size in REF_SIZES_PER_DATASET:
    for det_name in VIEW_PATHS:
        result_dir = (
            REPO_DIR
            / "logs"
            / "reproduction"
            / "ref_size"
            / f"size_{ref_size}"
            / det_name
        )
        df = compute_cte_metrics(result_dir, MAIN_KEYS)
        for _, r in df.iterrows():
            ref_size_records.append(
                {
                    "ref_size": ref_size,
                    "detector": r["detector"],
                    "dataset": r["dataset"],
                    "method": r["method"],
                    "accuracy": r["accuracy"],
                    "f1": r["f1"],
                }
            )

ref_size_df = pd.DataFrame(ref_size_records)

if ref_size_df.empty:
    print("No reference size results collected.")
else:
    # ── Plot: accuracy vs reference size ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    detectors = sorted(ref_size_df["detector"].unique())
    methods_to_plot = {
        "xg": ("MoSEs-xg", "tab:blue", "o"),
        "logistic": ("MoSEs-lr", "tab:green", "s"),
        "full_constant": ("Static Threshold", "tab:red", "^"),
        "full_count": ("Nearest Voting", "tab:orange", "D"),
    }

    for ax, det in zip(axes, detectors):
        for method, (label, color, marker) in methods_to_plot.items():
            subset = ref_size_df[
                (ref_size_df["detector"] == det) & (ref_size_df["method"] == method)
            ]
            if subset.empty:
                continue
            avg = subset.groupby("ref_size")["accuracy"].mean()
            ax.plot(
                avg.index,
                avg.values,
                marker=marker,
                color=color,
                label=label,
                linewidth=2,
            )
        ax.set_title(det)
        ax.set_xlabel("Reference samples per dataset")
        if ax == axes[0]:
            ax.set_ylabel("Avg. Accuracy (main datasets)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle("Detection Accuracy vs. Reference Set Size", fontsize=14, y=1.02)
    fig.tight_layout()
    plot_path = RESULTS_ROOT / "ref_size_sensitivity.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved to: {plot_path}")

    # ── Summary table ──
    summary = ref_size_df[
        ref_size_df["method"].isin(["xg", "full_constant"])
    ].pivot_table(
        index=["detector", "method"],
        columns="ref_size",
        values="accuracy",
        aggfunc="mean",
    )
    summary.columns = [f"N={c}" for c in summary.columns]
    print(f"\n{'='*80}")
    print("  Reference Size Sensitivity — Avg. Accuracy across main datasets")
    print(f"{'='*80}")
    print(summary.to_string(float_format=lambda x: f"{x:.4f}"))

    ref_size_csv = RESULTS_ROOT / "ref_size_sensitivity.csv"
    ref_size_df.to_csv(ref_size_csv, index=False)
    print(f"\nDetailed results saved to: {ref_size_csv}")

sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 11: Additional Experiment 2 — Text Length Stratified Evaluation
# =========================
# CTE uses text length (cond[0]) as a conditional feature. We bucket test
# samples by text length and check whether MoSEs' advantage is concentrated
# on certain length ranges. If MoSEs underperforms on very short texts,
# that is an important practical finding.

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def load_test_cond_field(test_json_path, field_idx):
    """Load a specific conditional feature (by index) from a processed test JSON."""
    with open(test_json_path, "r") as f:
        data = json.load(f)
    return [item["cond"][field_idx] for item in data]


def load_cte_result_json(result_dir, test_filename):
    """Load CTE result JSON corresponding to a given test file."""
    stem = Path(test_filename).stem  # e.g., "cmv_dataset"
    result_path = Path(result_dir) / f"{stem}_results.json"
    if result_path.exists():
        with open(result_path, "r") as f:
            return json.load(f)
    return None


# ── Collect per-sample predictions matched with text lengths ──
length_records = []

for det_name, views in VIEW_PATHS.items():
    main_test_dir = Path(views["main_test"])
    result_dir = REPO_DIR / "logs" / "reproduction" / "main" / det_name

    for test_file in sorted(main_test_dir.glob("*.json")):
        ds_key = dataset_key_from_json_name(test_file.name)
        if ds_key not in MAIN_KEYS:
            continue

        # Text length is cond[0]
        lengths = load_test_cond_field(test_file, 0)

        # CTE results
        results = load_cte_result_json(result_dir, test_file.name)
        if results is None:
            print(f"WARNING: No CTE results for {det_name}/{ds_key}, skipping.")
            continue

        for method in ["xg", "logistic", "full_constant"]:
            if method not in results:
                continue
            y_true = results[method]["y_true"]
            y_pred = results[method]["y_pred"]

            assert len(lengths) == len(y_true), (
                f"Length mismatch: {len(lengths)} lengths vs {len(y_true)} predictions "
                f"for {det_name}/{ds_key}/{method}"
            )

            for i in range(len(lengths)):
                length_records.append(
                    {
                        "detector": det_name,
                        "dataset": ds_key,
                        "method": method,
                        "text_length": lengths[i],
                        "y_true": y_true[i],
                        "y_pred": y_pred[i],
                        "correct": int(y_true[i] == y_pred[i]),
                    }
                )

length_df = pd.DataFrame(length_records)

if length_df.empty:
    print("No data collected. Run Cell 6 (CTE evaluation) first.")
else:
    # ── Create text length bins (terciles across all samples) ──
    length_df["length_bin"] = pd.qcut(
        length_df["text_length"], q=3, labels=["Short", "Medium", "Long"]
    )

    bin_edges = pd.qcut(length_df["text_length"], q=3, retbins=True)[1]
    print(f"Text length tercile boundaries: {[f'{x:.0f}' for x in bin_edges]}")
    print(
        f"  Short:  <= {bin_edges[1]:.0f} tokens\n"
        f"  Medium: {bin_edges[1]:.0f} – {bin_edges[2]:.0f} tokens\n"
        f"  Long:   > {bin_edges[2]:.0f} tokens"
    )

    METHOD_DISPLAY = {
        "xg": "MoSEs-xg",
        "full_constant": "Static Threshold",
        "logistic": "MoSEs-lr",
    }
    bins = ["Short", "Medium", "Long"]

    # ── Table: accuracy per length bin, detector, method ──
    print(f"\n{'='*80}")
    print("  Text Length Stratified Accuracy (main datasets)")
    print(f"{'='*80}")

    strat_rows = []
    for det in sorted(length_df["detector"].unique()):
        for method, label in METHOD_DISPLAY.items():
            for bin_name in bins:
                subset = length_df[
                    (length_df["detector"] == det)
                    & (length_df["method"] == method)
                    & (length_df["length_bin"] == bin_name)
                ]
                if subset.empty:
                    continue
                strat_rows.append(
                    {
                        "Detector": det,
                        "Method": label,
                        "Length Bin": bin_name,
                        "Accuracy": subset["correct"].mean(),
                        "N": len(subset),
                    }
                )

    strat_table = pd.DataFrame(strat_rows)
    print(strat_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # ── Plot: grouped bar chart per detector ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    detectors = sorted(length_df["detector"].unique())
    x = np.arange(len(bins))
    width = 0.25

    for ax, det in zip(axes, detectors):
        for i, (method, label) in enumerate(METHOD_DISPLAY.items()):
            accs = []
            for b in bins:
                subset = length_df[
                    (length_df["detector"] == det)
                    & (length_df["method"] == method)
                    & (length_df["length_bin"] == b)
                ]
                accs.append(subset["correct"].mean() if not subset.empty else 0)
            ax.bar(x + i * width, accs, width, label=label)

        ax.set_title(det)
        ax.set_xlabel("Text Length Bin")
        if ax == axes[0]:
            ax.set_ylabel("Accuracy")
        ax.set_xticks(x + width)
        ax.set_xticklabels(bins)
        ax.legend(fontsize=8)
        ax.set_ylim(0.5, 1.05)
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle(
        "Detection Accuracy by Text Length (Main Datasets)", fontsize=14, y=1.02
    )
    fig.tight_layout()
    plot_path = RESULTS_ROOT / "text_length_stratified.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nPlot saved to: {plot_path}")

    # ── MoSEs-xg improvement over Static Threshold per bin ──
    print(f"\n{'='*80}")
    print("  MoSEs-xg Improvement over Static Threshold by Text Length")
    print(f"{'='*80}")

    imp_rows = []
    for det in sorted(length_df["detector"].unique()):
        for bin_name in bins:
            xg_sub = length_df[
                (length_df["detector"] == det)
                & (length_df["method"] == "xg")
                & (length_df["length_bin"] == bin_name)
            ]
            st_sub = length_df[
                (length_df["detector"] == det)
                & (length_df["method"] == "full_constant")
                & (length_df["length_bin"] == bin_name)
            ]
            if xg_sub.empty or st_sub.empty:
                continue
            xg_acc = xg_sub["correct"].mean()
            st_acc = st_sub["correct"].mean()
            delta = xg_acc - st_acc
            pct = (delta / st_acc * 100) if st_acc > 0 else 0
            imp_rows.append(
                {
                    "Detector": det,
                    "Length Bin": bin_name,
                    "MoSEs-xg": xg_acc,
                    "Static Thresh.": st_acc,
                    "Delta": delta,
                    "Improv. %": pct,
                }
            )

    imp_table = pd.DataFrame(imp_rows)
    print(imp_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # Save
    strat_csv = RESULTS_ROOT / "text_length_stratified.csv"
    length_df.to_csv(strat_csv, index=False)
    print(f"\nDetailed data saved to: {strat_csv}")

sync_outputs_to_drive()